# MedGemma Augmentation on Colab

이 노트북은 `LoRA_ASR/augment_medgemma.py`를 Google Colab에서 실행하기 위한 최소 셋업입니다.

사전 준비:
- Colab 런타임을 GPU로 변경
- Hugging Face access token을 Colab Secret `HF_TOKEN`으로 등록
- `raw_medical_data.jsonl`을 Google Drive에 업로드

In [ ]:
REPO_URL = "https://github.com/jhparktime/ASR_finetune.git"
REPO_DIR = "/content/ASR_finetune"

!git clone {REPO_URL} {REPO_DIR}
%cd /content/ASR_finetune/LoRA_ASR

In [ ]:
!python -m pip install -U pip
!pip install -r requirements-colab.txt

In [ ]:
import os
from google.colab import drive

drive.mount("/content/drive")

try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN loaded from Colab Secret")
except Exception as exc:
    print("HF_TOKEN secret could not be loaded:", exc)
    print("필요하면 아래 줄처럼 직접 설정하세요: os.environ['HF_TOKEN'] = 'hf_xxx'")

os.environ.setdefault("HF_HOME", "/content/hf_home")

In [ ]:
INPUT_FILE = "/content/drive/MyDrive/medgemma/raw_medical_data.jsonl"
OUTPUT_FILE = "/content/drive/MyDrive/medgemma/augmented_medical_medgemma.jsonl"
MODEL_NAME = "google/medgemma-27b-text-it"
LIMIT = 1
BATCH_SIZE = 1
MAX_NEW_TOKENS = 512
LOAD_IN_4BIT = True
LOAD_IN_8BIT = False

assert not (LOAD_IN_4BIT and LOAD_IN_8BIT), "4bit와 8bit는 동시에 켤 수 없습니다."

In [ ]:
import subprocess

cmd = [
    "python",
    "augment_medgemma.py",
    "--input-file", INPUT_FILE,
    "--output-file", OUTPUT_FILE,
    "--model-name", MODEL_NAME,
    "--batch-size", str(BATCH_SIZE),
    "--max-new-tokens", str(MAX_NEW_TOKENS),
    "--device-map", "auto",
    "--overwrite",
]

if LIMIT is not None:
    cmd.extend(["--limit", str(LIMIT)])
if LOAD_IN_4BIT:
    cmd.append("--load-in-4bit")
if LOAD_IN_8BIT:
    cmd.append("--load-in-8bit")

print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)